In [5]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
from obstore.store import S3Store
import pandas as pd
from pyproj import Transformer
from tqdm import tqdm
import zarr
from zarr.storage import ObjectStore

# -------------------------------------------------------------------------
# 1. Connect to Source Cooperative Cloud Store
# -------------------------------------------------------------------------
zarr.config.set({"array.sharding_coalesce_max_gap_bytes": 0})

s3 = S3Store.from_url(
    "s3://olmoearth/olmoearth-v1-3-embeddings/embeddings.zarr",
    endpoint="https://data.source.coop",
    skip_signature=True,
)
zarr_store = ObjectStore(s3, read_only=True)
root = zarr.open_group(store=zarr_store, mode="r")

print("Connected to OlmoEarth v1.3 Zarr archive successfully!")


def dequantize(q):
  q = np.asarray(q)
  v = np.where(q == -128, np.nan, q).astype("float32")
  return (np.abs(v) / 127.5) ** 2.0 * np.sign(v)


# -------------------------------------------------------------------------
# 2. Input Dataset Configuration
# -------------------------------------------------------------------------
study_folder = Path("output" / "germany_barw_bbch51_2017-01-01_2025-12-31")
input_csv = study_folder / "curated_for_modeling_with_presto_embeddings.csv"
if not input_csv.exists():
  input_csv = list(study_folder.glob("curated*.csv"))[0]

df_base = pd.read_csv(input_csv, sep=";")
N = len(df_base)
print(f"Loaded {N} observations from {input_csv.name}")
dim = 128

# -------------------------------------------------------------------------
# 3. Cache UTM Zone Transformers & Affine Matrices
# -------------------------------------------------------------------------
print("Pre-caching zone affine transforms...")
utm_meta = {}
for zone_num in [32, 33]:
  z_name = f"utm{zone_num:02d}"
  if z_name in root:
    zone = root[z_name]
    a, _, c, _, e, f = zone.attrs["spatial:transform"]
    utm_meta[zone_num] = {
        "transformer": Transformer.from_crs(
            "EPSG:4326", zone.attrs["proj:code"], always_xy=True
        ),
        "affine": (a, c, e, f),
        "time": np.array(zone["time"]),
        "emb_ds": zone["embeddings"],
    }

# -------------------------------------------------------------------------
# 4. Resolve Affine Pixel Indices
# -------------------------------------------------------------------------
print("Resolving pixel indices via spatial transform...")
query_tasks = []

for idx, row in df_base.iterrows():
  lat = float(row["Latitude"])
  lon = float(row["Longitude"])
  yr = int(row["Year"])

  zone_num = int((lon + 180) // 6) + 1

  if zone_num not in utm_meta:
    query_tasks.append(None)
    continue

  meta = utm_meta[zone_num]
  if yr not in meta["time"]:
    query_tasks.append(None)
    continue

  time_idx = int(np.where(meta["time"] == yr)[0][0])

  # Accurate WGS84 -> UTM transformation
  ux, uy = meta["transformer"].transform(lon, lat)

  # Exact affine mapping to 10m raster indices
  a, c, e, f = meta["affine"]
  col_idx = int((ux - c) / a)
  row_idx = int((uy - f) / e)

  query_tasks.append((zone_num, time_idx, row_idx, col_idx))


# -------------------------------------------------------------------------
# 5. Concurrent Point Extraction (128 Dimensions)
# -------------------------------------------------------------------------
def fetch_128d(task):
  if task is None:
    return np.full(dim, np.nan, dtype=np.float32)
  zone_num, time_idx, row_idx, col_idx = task
  try:
    ds = utm_meta[zone_num]["emb_ds"]
    q_vec = ds[time_idx, :dim, row_idx, col_idx]
    return dequantize(q_vec)
  except Exception:
    return np.full(dim, np.nan, dtype=np.float32)


print(f"Streaming 128-d OlmoEarth embeddings across {N} observations...")
with ThreadPoolExecutor(max_workers=20) as executor:
  results = list(
      tqdm(
          executor.map(fetch_128d, query_tasks),
          total=N,
          desc="OlmoEarth 128d",
      )
  )

olmo_matrix = np.vstack(results)

# Diagnostics
nan_count = np.isnan(olmo_matrix).sum()
print(f"NaN elements before median imputation: {nan_count}")

if np.isnan(olmo_matrix).any():
  col_medians = np.nanmedian(olmo_matrix, axis=0)
  col_medians = np.nan_to_num(col_medians, nan=0.0)
  inds = np.where(np.isnan(olmo_matrix))
  olmo_matrix[inds] = np.take(col_medians, inds[1])

print(f"Final matrix norm mean: {np.linalg.norm(olmo_matrix, axis=1).mean():.2f}")
print(f"Matrix shape: {olmo_matrix.shape}")

# -------------------------------------------------------------------------
# 6. Save Curated Modeling File
# -------------------------------------------------------------------------
olmo_cols = [f"olmo_{i}" for i in range(dim)]
df_olmo = pd.DataFrame(
    olmo_matrix, columns=olmo_cols, index=df_base.index
)

base_cols = [c for c in df_base.columns if not c.startswith("presto_")]
df_final = pd.concat([df_base[base_cols], df_olmo], axis=1)
output_path = "output" / "curated_for_modeling_with_olmo_embeddings.csv"
df_final.to_csv(output_path, sep=";", index=False)

print(f"\nSaved {output_path.name} with {dim} OlmoEarth dimensions!")
print(f"Path: {output_path.resolve()}")

Connected to OlmoEarth v1.3 Zarr archive successfully!


TypeError: unsupported operand type(s) for /: 'str' and 'str'